# Softmax Gate · 2K Colab render

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lroe/softmax/blob/main/render_2k_colab.ipynb)

This notebook renders the **19:54 Softmax Gate explainer** at **2560 × 1440 (2K)** with Remotion and writes every output to **Google Drive**.

- **Access first:** step 1 asks for Google Drive permission before anything is written.
- **Resumable:** every 20-second chunk is copied to Drive. If Colab disconnects, rerun the notebook; finished chunks are skipped.
- **Live progress:** the renderer streams frame counts, chunk numbers, speed, and ETA into the progress bar.
- **No GPU needed.** Chrome renders on the CPU. A free Colab runtime works, but a full 2K render can take several hours spread across sessions.
- **Source:** https://github.com/lroe/softmax
- **Not included:** `skill.md` is intentionally excluded from the repository.

> **First run:** `Runtime → Run all`, approve the Google Drive prompt, and let the last cell run. Interrupt the last cell to stop; completed chunks stay safe on Drive.


In [ ]:
# ---------------------------------------------------------------------------
# 1) Settings
# ---------------------------------------------------------------------------
REPO_URL = "https://github.com/lroe/softmax.git"
BRANCH = "main"
DRIVE_FOLDER = "SoftmaxGate2K"

TARGET_WIDTH = 2560          # 2K UHD
TARGET_HEIGHT = 1440
CHUNK_FRAMES = 600           # 20 s at 30 fps; copied to Drive after every chunk
CONCURRENCY = 0              # 0 = pick from CPU count
CRF = 16                     # lower = better quality / larger file

OUTPUT_NAME = f"softmax-gate-{TARGET_WIDTH}x{TARGET_HEIGHT}.mp4"
SCALE = TARGET_WIDTH / 1920

import os
from pathlib import Path

assert TARGET_WIDTH % 2 == 0 and TARGET_HEIGHT % 2 == 0, "Video dimensions must be even."
assert abs(TARGET_HEIGHT - 1080 * SCALE) < 0.01, "Keep the 16:9 aspect ratio."
if CONCURRENCY <= 0:
    CONCURRENCY = max(1, min(4, os.cpu_count() or 2))

REPO_DIR = Path("/content/softmax")
VIDEO_DIR = REPO_DIR / "video"
WORK_DIR = Path("/content/render-work")
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER

print(f"Target       : {TARGET_WIDTH} × {TARGET_HEIGHT} (scale {SCALE:.6f})")
print(f"Chunk size   : {CHUNK_FRAMES} frames")
print(f"Concurrency  : {CONCURRENCY}")
print(f"Drive output : {DRIVE_ROOT}")
print(f"Local work   : {WORK_DIR}")


## 1 · Google Drive access

This is the only place the notebook asks for permission. The next cell opens Google's authorization dialog. Nothing is written to Drive until you approve it.

Files will live in `MyDrive/SoftmaxGate2K/`:

```
SoftmaxGate2K/
├── softmax-gate-2560x1440.mp4
├── chunks/                 # resumable 2K segments
├── stills/                 # 2K test frame
├── logs/                   # renderer event log
├── render_state.json       # completed chunk bookkeeping
└── render_metadata.json    # final ffprobe summary
```


In [ ]:
from google.colab import drive
import shutil

MOUNT = Path("/content/drive")
if (MOUNT / "MyDrive").exists():
    print("Google Drive is already mounted.")
else:
    print("Requesting Google Drive access...")
    drive.mount(str(MOUNT), force_remount=False)
    print("Access granted.")

assert (MOUNT / "MyDrive").exists(), "Google Drive was not mounted."

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
for sub in ("chunks", "logs", "stills"):
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

local = shutil.disk_usage("/content")
print(f"\nLocal disk free : {local.free / 1e9:.1f} GB")
try:
    remote = shutil.disk_usage(DRIVE_ROOT)
    print(f"Drive space free: {remote.free / 1e9:.1f} GB")
except OSError:
    pass
print(f"Output folder   : {DRIVE_ROOT}")

(DRIVE_ROOT / "colab_render_config.json").write_text(
    f'{{"width": {TARGET_WIDTH}, "height": {TARGET_HEIGHT}, "chunk_frames": {CHUNK_FRAMES}, '
    f'"crf": {CRF}, "concurrency": {CONCURRENCY}}}\n'
)
print("Drive is ready.")


## 2 · Install Node.js, ffmpeg and Chrome libraries

Colab usually has ffmpeg. This cell adds Node 22 (if needed) and the shared libraries that Remotion's headless Chrome expects. It can take a minute on the first run.


In [ ]:
import os
import subprocess


def run_stream(cmd, cwd=None, check=True, env=None):
    """Run a command and print every line as it arrives."""
    printable = " ".join(str(x) for x in cmd)
    print(f"$ {printable}", flush=True)
    merged_env = {**os.environ, "DEBIAN_FRONTEND": "noninteractive", **(env or {})}
    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=cwd,
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {printable}")
    return code


def node_major():
    try:
        raw = subprocess.check_output(["node", "--version"], text=True).strip()
        return int(raw.lstrip("v").split(".")[0])
    except Exception:
        return 0


print("Updating apt and installing the base tools...")
run_stream(["apt-get", "update", "-qq"])
run_stream([
    "apt-get", "install", "-y", "-qq",
    "ffmpeg", "git", "curl", "ca-certificates",
    "fonts-dejavu-core", "fonts-liberation",
])

if node_major() < 20:
    print("\nInstalling Node.js 22...")
    run_stream(["bash", "-lc", "curl -fsSL https://deb.nodesource.com/setup_22.x | bash -"])
    run_stream(["apt-get", "install", "-y", "-qq", "nodejs"])

    # Colab images sometimes ship an older Node in /usr/local/bin, which comes
    # before /usr/bin on PATH. Point the local shims at the NodeSource install.
    if node_major() < 20:
        for name in ("node", "npm", "npx"):
            src = Path("/usr/bin") / name
            dst = Path("/usr/local/bin") / name
            if src.exists():
                if dst.exists() or dst.is_symlink():
                    dst.unlink()
                dst.symlink_to(src)

if node_major() < 20:
    raise RuntimeError("Node.js 20+ is required. Restart the runtime and rerun this cell.")

print("\nNode:", subprocess.check_output(["node", "--version"], text=True).strip())
print("npm :", subprocess.check_output(["npm", "--version"], text=True).strip())

# Shared libraries used by Chrome Headless Shell. Colab has many of these already.
def apt_has(package):
    return subprocess.run(
        ["apt-cache", "show", package],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    ).returncode == 0

sound_pkg = "libasound2" if apt_has("libasound2") else "libasound2t64"
chrome_libs = [
    "libnss3", "libnspr4", "libatk1.0-0", "libatk-bridge2.0-0", "libcups2",
    "libdrm2", "libxkbcommon0", "libxcomposite1", "libxdamage1", "libxfixes3",
    "libxrandr2", "libgbm1", sound_pkg, "libpango-1.0-0", "libcairo2",
    "libx11-xcb1", "libxcursor1", "libxi6", "libxtst6", "libatspi2.0-0",
    "libxshmfence1",
]
rc = run_stream(["apt-get", "install", "-y", "-qq", *chrome_libs], check=False)
if rc != 0:
    print("\nWarning: some optional Chrome libraries did not install. The frame test below will confirm whether rendering works.")


## 3 · Clone or update the project

This always checks out the latest `main` from GitHub. A fresh clone includes the 51 MB narration WAV, so the first clone can take a moment.


In [ ]:
import shutil
import subprocess

print("Fetching the latest project from GitHub...")
if (REPO_DIR / ".git").exists():
    run_stream(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH])
    run_stream(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"])
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run_stream([
        "git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)
    ])

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "log", "-1", "--format=%h  %ad  %s", "--date=iso"],
    text=True,
).strip()
print("\nChecked out:", commit)

assert (VIDEO_DIR / "package.json").exists(), "video/package.json is missing."
assert (VIDEO_DIR / "scripts" / "render-2k.mjs").exists(), "The 2K renderer is missing."
assert (VIDEO_DIR / "src" / "index.ts").exists()

audio = VIDEO_DIR / "public" / "audio.wav"
audio_mb = audio.stat().st_size / 1e6
print(f"Narration    : {audio_mb:.1f} MB")
assert audio_mb > 40, "audio.wav is missing or truncated."
print("Project is ready.")


## 4 · Install JavaScript dependencies

`npm ci` installs the exact Remotion version from `package-lock.json`. The cell remembers the lockfile hash, so rerunning it in the same runtime is instant. It also downloads the Chrome Headless Shell used for rendering.


In [ ]:
import hashlib
import subprocess

lock = VIDEO_DIR / "package-lock.json"
stamp = VIDEO_DIR / ".install-stamp"
node_version = subprocess.check_output(["node", "--version"], text=True).strip()
expected = hashlib.sha256(lock.read_bytes()).hexdigest() + "|" + node_version

if stamp.exists() and stamp.read_text().strip() == expected:
    print("node_modules already matches package-lock.json.")
else:
    run_stream(["npm", "ci", "--no-audit", "--no-fund"], cwd=VIDEO_DIR)
    stamp.write_text(expected + "\n")

remotion_version = subprocess.check_output(
    ["node", "-e", "console.log(require('remotion/package.json').version)"],
    cwd=VIDEO_DIR,
    text=True,
).strip()
print(f"\nRemotion {remotion_version}")

print("\nEnsuring Remotion has a headless browser...")
run_stream(["npx", "remotion", "browser", "ensure"], cwd=VIDEO_DIR)


## 5 · Verify a 2K frame before the long render

This renders frame 150 at 2560 × 1440, checks the pixel dimensions, saves it to Drive, and shows it inline. If this frame looks right, the full render will too.


In [ ]:
import shutil

from PIL import Image
from IPython.display import Image as IPImage, display

WORK_DIR.mkdir(parents=True, exist_ok=True)
STILL_FRAME = 150
still_local = WORK_DIR / f"still_{STILL_FRAME:05d}.png"
still_drive = DRIVE_ROOT / "stills" / f"still_{STILL_FRAME:05d}_{TARGET_WIDTH}x{TARGET_HEIGHT}.png"

print(f"Rendering frame {STILL_FRAME} at {TARGET_WIDTH} × {TARGET_HEIGHT}...")
run_stream([
    "npx", "remotion", "still",
    "src/index.ts", "SoftmaxGate", str(still_local),
    "--frame", str(STILL_FRAME),
    "--scale", f"{SCALE:.17g}",
    "--image-format", "png",
    "--gl", "swangle",
    "--overwrite",
], cwd=VIDEO_DIR)

im = Image.open(still_local)
print(f"\nRendered size: {im.width} × {im.height}")
assert (im.width, im.height) == (TARGET_WIDTH, TARGET_HEIGHT), "2K frame has the wrong dimensions."

shutil.copy2(still_local, still_drive)
print(f"Saved to Drive: {still_drive}")
display(IPImage(filename=str(still_drive), width=960))


## 6 · Render the full video with live progress

The renderer splits the timeline into 600-frame chunks (20 s at 30 fps), renders each one at 2K, and copies it to Drive before starting the next. The progress bar tracks all 35,847 frames; the status text shows the current chunk, speed, and ETA.

- **Stop safely:** interrupt this cell. Completed chunks stay on Drive.
- **Resume:** rerun the notebook from the top. Completed chunks are skipped automatically.
- **Time:** depends on the Colab runtime. A free 2-CPU runtime is roughly 6–12 hours for the whole video; faster runtimes finish sooner.


In [ ]:
import json
import shutil
import subprocess
from datetime import datetime

from IPython.display import FileLink, Image as IPImage, display
from tqdm.auto import tqdm

WORK_DIR.mkdir(parents=True, exist_ok=True)
log_path = WORK_DIR / f"render_events_{datetime.now():%Y%m%d_%H%M%S}.jsonl"
log_handle = open(log_path, "w", encoding="utf-8")

cmd = [
    "node", "scripts/render-2k.mjs",
    "--output-dir", str(DRIVE_ROOT),
    "--work-dir", str(WORK_DIR),
    "--output-name", OUTPUT_NAME,
    "--chunk", str(CHUNK_FRAMES),
    "--concurrency", str(CONCURRENCY),
    "--crf", str(CRF),
    "--scale", f"{SCALE:.17g}",
]

print("$", " ".join(cmd))
print("Finished chunks are copied to Google Drive before the next one starts.\n")

bar = None
stitch_bar = None
final = None
bundle_bucket = -1
proc = None

try:
    proc = subprocess.Popen(
        cmd,
        cwd=VIDEO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for raw in proc.stdout:
        line = raw.rstrip("\n")
        log_handle.write(line + "\n")
        log_handle.flush()

        if not line.startswith("@@RENDER@@ "):
            if line.strip():
                print(line, flush=True)
            continue

        event = json.loads(line[len("@@RENDER@@ "):])
        kind = event.get("type")

        if kind == "browser_ready":
            print("Headless Chrome ready.")
        elif kind == "bundle_start":
            print("Bundling the Remotion project...")
        elif kind == "bundle_progress":
            bucket = int(event["percent"] // 25)
            if bucket > bundle_bucket:
                bundle_bucket = bucket
                print(f"  bundling {event['percent']:.0f}%")
        elif kind == "bundle_done":
            print("Bundle ready.")
        elif kind == "composition_ready":
            total = event["totalFrames"]
            print(
                f"Composition: {event['outputWidth']} × {event['outputHeight']}, "
                f"{total:,} frames ({total / 30 / 60:.1f} min at 30 fps)"
            )
            bar = tqdm(
                total=total,
                unit=" frame",
                dynamic_ncols=True,
                smoothing=0.05,
                bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]",
            )
        elif kind == "plan":
            print(f"Plan: {event['totalChunks']} chunks of {event['chunkFrames']} frames.\n")
        elif kind == "chunk_start":
            if bar is not None:
                bar.set_description(f"chunk {event['index'] + 1}/{event['chunkCount']} {event['label']}")
        elif kind == "progress":
            if bar is not None:
                bar.n = max(bar.n, min(event["globalRendered"], bar.total))
                bar.set_postfix_str(
                    f"{event['fps']:.2f} fps | {event['chunkRendered']}/{event['chunkFrames']} in chunk"
                )
                bar.refresh()
        elif kind == "chunk_skip":
            if bar is not None:
                bar.n = max(bar.n, event["completedFrames"])
                bar.refresh()
            tqdm.write(
                f"skip  chunk {event['index'] + 1}/{event['chunkCount']} "
                f"({event['label']}) already on Drive"
            )
        elif kind == "chunk_done":
            if bar is not None:
                bar.n = max(bar.n, event["completedFrames"])
                bar.refresh()
            tqdm.write(
                f"done  chunk {event['index'] + 1}/{event['chunkCount']} "
                f"({event['frames']} frames, {event['seconds'] / 60:.1f} min, "
                f"{event['bytes'] / 1e6:.1f} MB)"
            )
        elif kind == "copy_start":
            if bar is not None:
                bar.set_description(f"copying {event['label']}")
        elif kind == "copy_progress":
            if bar is not None:
                bar.set_postfix_str(f"copy {event['percent']:.0f}%")
        elif kind in ("copy_done", "copy_retry", "chunk_reuse", "chunk_rendered"):
            pass
        elif kind == "render_done":
            print(f"\nAll chunks are on Drive ({event['completedFrames']:,} frames).")
        elif kind == "stitch_start":
            if bar is not None:
                bar.close()
                bar = None
            print(f"\nStitching {event['chunks']} chunks and the original audio...")
            stitch_bar = tqdm(total=100, unit="%", desc="Stitching", dynamic_ncols=True)
        elif kind == "stitch_progress":
            if stitch_bar is not None:
                stitch_bar.n = min(99, int(event["percent"]))
                stitch_bar.refresh()
        elif kind == "stitch_done":
            if stitch_bar is not None:
                stitch_bar.n = 100
                stitch_bar.refresh()
                stitch_bar.close()
                stitch_bar = None
            print("Stitch complete.")
        elif kind == "metadata":
            final = event
            print(
                f"\nFinal: {event['width']} × {event['height']}, "
                f"{event['durationSeconds'] / 60:.2f} min, {event['bytes'] / 1e9:.2f} GB"
            )
        elif kind == "poster":
            print(f"Poster: {event['path']}")
        elif kind == "done":
            final = {**(final or {}), **event}
        elif kind == "error":
            raise RuntimeError(event.get("message", "The renderer failed."))
        elif kind == "log":
            print(event.get("message", ""))

    exit_code = proc.wait()
    if exit_code != 0:
        raise RuntimeError(f"The renderer exited with code {exit_code}. See {log_path}.")

except KeyboardInterrupt:
    print("\nInterrupted. Stopping the current chunk; completed chunks remain on Drive.")
    if proc is not None and proc.poll() is None:
        proc.terminate()
    raise

finally:
    log_handle.close()
    if bar is not None:
        bar.close()
    if stitch_bar is not None:
        stitch_bar.close()
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=15)
        except subprocess.TimeoutExpired:
            proc.kill()
    try:
        shutil.copy2(log_path, DRIVE_ROOT / "logs" / log_path.name)
        print(f"\nEvent log saved to {DRIVE_ROOT / 'logs' / log_path.name}")
    except Exception as exc:
        print(f"\nCould not copy the event log to Drive: {exc}")

if final is None:
    raise RuntimeError("The renderer finished without a final video record. Check the event log.")

print(f"\n2K render finished in {final.get('humanTime', 'unknown')}.")
print(f"File: {final.get('output', DRIVE_ROOT / OUTPUT_NAME)}")
display(FileLink(str(final.get("output", DRIVE_ROOT / OUTPUT_NAME)), result_html_prefix="Open the 2K MP4: "))

poster = DRIVE_ROOT / "poster_2560x1440.jpg"
if poster.exists():
    display(IPImage(filename=str(poster), width=960))


## 7 · Optional housekeeping

The final MP4 is independent of the chunk files. Once you have downloaded and played the final file, you can delete `chunks/` to free Drive space. Leave `DELETE_CHUNKS = False` until then.


In [ ]:
import subprocess

DELETE_CHUNKS = False

verify = DRIVE_ROOT / OUTPUT_NAME
if verify.exists():
    print(f"Final file: {verify}")
    print(f"Size      : {verify.stat().st_size / 1e9:.2f} GB")
    probe = subprocess.check_output([
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "stream=width,height,r_frame_rate",
        "-of", "default=noprint_wrappers=1", str(verify),
    ], text=True)
    print(probe.strip())
else:
    print("Final file not found yet. Run the render cell first.")

if DELETE_CHUNKS and verify.exists():
    chunks = sorted((DRIVE_ROOT / "chunks").glob("chunk_*.mp4"))
    freed = sum(p.stat().st_size for p in chunks)
    for p in chunks:
        p.unlink()
    print(f"Deleted {len(chunks)} chunk files ({freed / 1e9:.2f} GB freed).")
elif DELETE_CHUNKS:
    print("Refusing to delete chunks because the final MP4 is missing.")


## Troubleshooting

| Symptom | Fix |
| --- | --- |
| `Google Drive was not mounted` | Rerun step 1 and approve the browser popup. In Colab, popups can be blocked; allow them for `colab.research.google.com`. |
| `Chrome Headless Shell` will not start | Rerun step 2, then `npx remotion browser ensure` in step 4. The still test in step 5 will confirm. |
| Render stops when Colab disconnects | Rerun the notebook. Existing chunks are checked on Drive and skipped. |
| Drive runs out of space | Set `DELETE_CHUNKS = True` in step 7 after the final MP4 is verified. |
| Want a smaller file | Raise `CRF` in step 1 (for example 20–22) and rerun. Use `--force` only if you intend to re-render every chunk. |

The renderer is `video/scripts/render-2k.mjs`. It uses Remotion's `@remotion/renderer` API, so progress comes from actual rendered/encoded frame counts rather than parsed console output.
